# Filter pseudo-QA và train student trên Kaggle

Notebook dùng một file pseudo-QA đã có score, filter theo score field tuỳ chọn, ghép với A-OKVQA train và self-train BLIP student. Default vẫn là VQAScore (`scores.vqascore`) để tương thích notebook cũ.

Không chạy lại scorer/filter nặng và không tải lại A-OKVQA/COCO: notebook đọc các dataset đã attach trong `/kaggle/input`. Nếu có 2 GPU, training dùng DDP trên cả hai. Output nằm trong `/kaggle/working/student_<EXPERIMENT_NAME>`.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/fantastichaha11/SelTDA.git'
BRANCH = 'feat/pseudo-label-filter'
REPO_DIR = Path('/kaggle/working/SelTDA')
AOKVQA_INPUT_ROOT = Path('/kaggle/input/datasets/phong2004/a-okvqa/aokvqa')
AOKVQA_ROOT = Path('/kaggle/working/data/aokvqa')
COCO_INPUT_ROOT = Path('/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017')
EXPERIMENT_NAME = 'vqascore'
OUTPUT_DIR = Path(f'/kaggle/working/student_{EXPERIMENT_NAME}')
CONFIG_ARG = f'configs/aokvqa_{EXPERIMENT_NAME}_kaggle.yaml'
CONFIG_PATH = REPO_DIR / CONFIG_ARG

# SCORE_INPUT có thể là file scored-pool JSON hoặc folder score_cache.
# Nếu là folder cache, notebook tìm file theo rule trong filtering.score_cache: <raw_input_stem>__<gate>.json.
SCORE_INPUT = '/kaggle/input/datasets/phong2004/seltda/synthetic_data_vqascore_scored.json'
SCORE_INPUT_GLOB = '*_scored.json'
# Cần khi SCORE_INPUT là folder cache. Nếu None, notebook sẽ cố tìm raw JSON theo stem trong cache payload/input.
RAW_PSEUDO_JSON = None
SYNTH_IMAGE_ROOT = Path('/kaggle/input/datasets/mathew0george/coco-2017-unlabeled/unlabeled2017')

# Filter control:
# - FILTER_MODE='top_quantile': giữ top KEEP_TOP theo SCORE_FIELD.
# - FILTER_MODE='cascade': giữ record pass tất cả gate trong GATE_FILTERS.
# - FILTER_MODE='fusion': normalize nhiều gate, tính weighted average, giữ top FUSION_KEEP_TOP.
# - FILTER_MODE='all': dùng toàn bộ pseudo-QA trong SCORE_INPUT/RAW_PSEUDO_JSON, không cần score field.
# Ví dụ đổi experiment: set EXPERIMENT_NAME='g124', SCORE_INPUT='...', SCORE_FIELD='scores.<field_trong_json>'.
# Nếu input đã là file filtered sẵn, set FILTER_MODE='all' và đổi FILTERED_SYNTH_FILE theo tên muốn lưu.
FILTER_MODE = 'top_quantile'
SCORE_FIELD = 'scores.vqascore'
# Nếu None, notebook tự suy ra từ FILTER_MODE/GATE_FILTERS. Nếu dùng cache folder nhiều gate, có thể set rõ list này.
SCORE_FIELDS = None
SCORE_HIGHER_IS_BETTER = True
FILTERED_SYNTH_FILE = f'synthetic_data_{EXPERIMENT_NAME}'
KEEP_TOP = 0.75
GATE_FILTERS = [
    {'field': SCORE_FIELD, 'keep_top': KEEP_TOP, 'higher_is_better': SCORE_HIGHER_IS_BETTER},
    # Ví dụ multi-gate:
    # {'field': 'scores.xcons', 'keep_top': 0.75, 'higher_is_better': True},
    # {'field': 'scores.itm', 'keep_top': 0.75, 'higher_is_better': True},
]
FUSION_FIELDS = ['scores.vqascore', 'scores.xcons', 'scores.itm']
FUSION_WEIGHTS = {'vqascore': 1.0, 'xcons': 1.0, 'itm': 1.0}
FUSION_NORMALIZE = 'minmax'  # minmax | rank | none
FUSION_KEEP_TOP = KEEP_TOP
MAX_SYNTH_RECORDS = None
MAX_GPUS = 2
BATCH_SIZE_PER_GPU = 4
BATCH_SIZE_TEST = 4
MAX_EPOCH = 10
TRUNCATE_TRAIN_TO = 34000
SEED = 42

# Resume control:
# - AUTO_RESUME=True: tự resume checkpoint_*.pth mới nhất trong OUTPUT_DIR nếu có.
# - RESUME_CHECKPOINT='.../checkpoint_xx.pth': resume từ path cụ thể, ví dụ checkpoint attach qua Kaggle Input.
# - AUTO_RESUME=False và RESUME_CHECKPOINT=None: train lại từ pretrained.
AUTO_RESUME = True
RESUME_CHECKPOINT = None

In [ ]:
if not REPO_DIR.exists():
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
    if _exit_code != 0:
        raise RuntimeError(f'git clone failed với exit code {_exit_code}')
else:
    print('Repo đã tồn tại, bỏ qua clone:', REPO_DIR)

# Không dùng requirements.txt vì các pin NumPy cũ không phù hợp Kaggle Python 3.12.
%pip install -q --upgrade omegaconf==2.3.0 hydra-core==1.3.2 timm==0.4.12 fairscale==0.4.13 transformers==4.36.1
print('Clone và dependency setup hoàn tất.')

In [ ]:
import json, os
import numpy as np

score_input_path = Path(SCORE_INPUT) if SCORE_INPUT is not None else None
if score_input_path is None:
    candidates = sorted(Path('/kaggle/input').rglob(SCORE_INPUT_GLOB))
    if len(candidates) != 1:
        raise RuntimeError(
            'Không tự xác định được scored JSON/cache folder. Hãy đặt SCORE_INPUT ở cell cấu hình. '
            f'Candidates={candidates}'
        )
    score_input_path = candidates[0]

assert score_input_path.exists(), score_input_path
assert SYNTH_IMAGE_ROOT.is_dir(), (
    f'Không thấy SYNTH_IMAGE_ROOT={SYNTH_IMAGE_ROOT}. '
    'Hãy sửa path tới COCO unlabeled đã attach trên Kaggle.'
)

def get_nested(record, dotted_path):
    value = record
    for key in dotted_path.split('.'):
        value = value[key]
    return value

def score_value(record, score_field=None):
    return float(get_nested(record, score_field or SCORE_FIELD))

def required_score_fields():
    if SCORE_FIELDS is not None:
        return list(SCORE_FIELDS)
    if FILTER_MODE == 'fusion':
        return list(FUSION_FIELDS)
    if FILTER_MODE == 'cascade':
        return [spec['field'] for spec in GATE_FILTERS]
    if FILTER_MODE == 'all':
        return []
    return [SCORE_FIELD]

def record_key(record):
    # Same rule as filtering.score_cache.record_key.
    return f"{record.get('image', '')}\0{record.get('question', '')}"

def gate_from_score_field(score_field):
    parts = score_field.split('.')
    if len(parts) >= 2 and parts[0] == 'scores':
        return parts[1]
    raise ValueError(f'Không suy ra gate từ SCORE_FIELD={score_field!r}; dùng dạng scores.<gate>')

def normalize_values(values, mode='minmax'):
    arr = np.asarray(values, dtype=np.float64)
    if len(arr) == 0:
        return []
    if mode == 'none':
        return [float(v) for v in arr]
    if mode == 'rank':
        if len(arr) == 1:
            return [1.0]
        order = np.argsort(arr, kind='stable')
        ranks = np.empty(len(arr), dtype=np.float64)
        for rank, idx in enumerate(order):
            ranks[idx] = rank / (len(arr) - 1)
        return ranks.tolist()
    if mode != 'minmax':
        raise ValueError(f'FUSION_NORMALIZE không hợp lệ: {mode}')
    lo, hi = float(np.min(arr)), float(np.max(arr))
    if hi <= lo:
        return [0.0 for _ in arr]
    return ((arr - lo) / (hi - lo)).astype(float).tolist()

def fusion_score_from_normalized(normalized_scores, weights, gate_order):
    total = 0.0
    total_w = 0.0
    for gate in gate_order:
        w = float(weights.get(gate, 0.0))
        if w <= 0:
            continue
        total += w * float(normalized_scores.get(gate, 0.0))
        total_w += w
    return 0.0 if total_w <= 0 else total / total_w

def load_json(path):
    with Path(path).open(encoding='utf-8') as f:
        return json.load(f)

def find_raw_json_for_cache(cache_payload, cache_file, cache_dir):
    if RAW_PSEUDO_JSON is not None:
        raw_path = Path(RAW_PSEUDO_JSON)
        if not raw_path.is_file():
            raise FileNotFoundError(f'RAW_PSEUDO_JSON không tồn tại: {raw_path}')
        return raw_path

    input_name = Path(str(cache_payload.get('input', ''))).name
    candidates = []
    if input_name:
        candidates.extend(cache_dir.rglob(input_name))
        candidates.extend(Path('/kaggle/input').rglob(input_name))
    if not candidates:
        raw_stem = cache_file.name.rsplit('__', 1)[0]
        candidates.extend(cache_dir.rglob(f'{raw_stem}.json'))
        candidates.extend(Path('/kaggle/input').rglob(f'{raw_stem}.json'))
    candidates = sorted({p.resolve() for p in candidates if p.is_file() and '__' not in p.stem})
    if len(candidates) != 1:
        raise RuntimeError(
            'Không tự tìm được raw pseudo JSON để apply cache. '
            'Hãy set RAW_PSEUDO_JSON trong cell config. '
            f'Candidates={candidates}'
        )
    return candidates[0]

def load_records_from_score_input(score_input_path):
    if score_input_path.is_file():
        return load_json(score_input_path), score_input_path, []

    if not score_input_path.is_dir():
        raise FileNotFoundError(score_input_path)

    fields = required_score_fields()
    if not fields:
        if RAW_PSEUDO_JSON is None:
            raise ValueError('SCORE_INPUT là folder cache nhưng FILTER_MODE=all không có SCORE_FIELDS. Hãy set RAW_PSEUDO_JSON hoặc dùng file JSON.')
        return load_json(RAW_PSEUDO_JSON), Path(RAW_PSEUDO_JSON), []

    selected_cache_files = []
    for field in fields:
        gate = gate_from_score_field(field)
        if RAW_PSEUDO_JSON is not None:
            raw_stem = Path(RAW_PSEUDO_JSON).stem
            matches = sorted(score_input_path.rglob(f'{raw_stem}__{gate}.json'))
        else:
            matches = sorted(score_input_path.rglob(f'*__{gate}.json'))
        if not matches:
            raise FileNotFoundError(f'Không tìm thấy cache *__{gate}.json trong {score_input_path}')
        if len(matches) > 1:
            names = [str(p) for p in matches]
            raise RuntimeError(f'Tìm thấy nhiều cache cho gate={gate}: {names}. Hãy set RAW_PSEUDO_JSON để chọn đúng raw input.')
        selected_cache_files.append((field, gate, matches[0]))

    first_payload = load_json(selected_cache_files[0][2])
    raw_json_path = find_raw_json_for_cache(first_payload, selected_cache_files[0][2], score_input_path)
    records = load_json(raw_json_path)
    used = []
    for field, gate, cache_file in selected_cache_files:
        payload = load_json(cache_file)
        if payload.get('gate') != gate:
            raise ValueError(f'Cache gate mismatch: {cache_file} có gate={payload.get("gate")}, cần {gate}')
        scores = payload.get('scores')
        if not isinstance(scores, dict):
            raise ValueError(f'Cache thiếu scores dict: {cache_file}')
        applied = 0
        extras = payload.get('extras') or {}
        for record in records:
            key = record_key(record)
            if key not in scores:
                continue
            record.setdefault('scores', {})[gate] = float(scores[key])
            if isinstance(extras.get(key), dict):
                record.update(extras[key])
            applied += 1
        if applied != len(records):
            raise ValueError(f'Chỉ apply được {applied}/{len(records)} scores từ {cache_file}')
        used.append(cache_file)
    return records, raw_json_path, used

scored_records, scored_json_path, cache_files_used = load_records_from_score_input(score_input_path)

if FILTER_MODE == 'all':
    valid_records = list(scored_records)
    filtered_records = list(valid_records)
    threshold = None
elif FILTER_MODE == 'top_quantile':
    valid_records = []
    missing_score = 0
    for record in scored_records:
        try:
            score_value(record)
            valid_records.append(record)
        except (KeyError, TypeError, ValueError):
            missing_score += 1
    if missing_score:
        raise ValueError(f'{missing_score} records không có score hợp lệ tại SCORE_FIELD={SCORE_FIELD!r}')

    values = np.asarray([score_value(record) for record in valid_records], dtype=np.float64)
    if SCORE_HIGHER_IS_BETTER:
        threshold = float(np.quantile(values, 1.0 - KEEP_TOP))
        filtered_records = [record for record in valid_records if score_value(record) >= threshold]
    else:
        threshold = float(np.quantile(values, KEEP_TOP))
        filtered_records = [record for record in valid_records if score_value(record) <= threshold]
    filtered_records.sort(key=score_value, reverse=SCORE_HIGHER_IS_BETTER)
elif FILTER_MODE == 'cascade':
    valid_records = list(scored_records)
    thresholds = {}
    for spec in GATE_FILTERS:
        field = spec['field']
        keep_top = float(spec.get('keep_top', KEEP_TOP))
        higher = bool(spec.get('higher_is_better', True))
        missing_score = 0
        for record in valid_records:
            try:
                score_value(record, field)
            except (KeyError, TypeError, ValueError):
                missing_score += 1
        if missing_score:
            raise ValueError(f'{missing_score} records không có score hợp lệ tại {field!r}')
        values = np.asarray([score_value(record, field) for record in valid_records], dtype=np.float64)
        threshold_i = float(np.quantile(values, 1.0 - keep_top if higher else keep_top))
        thresholds[field] = threshold_i
    filtered_records = valid_records
    for spec in GATE_FILTERS:
        field = spec['field']
        higher = bool(spec.get('higher_is_better', True))
        threshold_i = thresholds[field]
        if higher:
            filtered_records = [record for record in filtered_records if score_value(record, field) >= threshold_i]
        else:
            filtered_records = [record for record in filtered_records if score_value(record, field) <= threshold_i]
    threshold = thresholds
    primary_field = GATE_FILTERS[0]['field']
    primary_higher = bool(GATE_FILTERS[0].get('higher_is_better', True))
    filtered_records.sort(key=lambda record: score_value(record, primary_field), reverse=primary_higher)
elif FILTER_MODE == 'fusion':
    valid_records = list(scored_records)
    fusion_fields = list(FUSION_FIELDS)
    fusion_gates = [gate_from_score_field(field) for field in fusion_fields]
    for field in fusion_fields:
        missing_score = 0
        for record in valid_records:
            try:
                score_value(record, field)
            except (KeyError, TypeError, ValueError):
                missing_score += 1
        if missing_score:
            raise ValueError(f'{missing_score} records không có score hợp lệ tại {field!r}')

    normed_by_gate = {}
    for field, gate in zip(fusion_fields, fusion_gates):
        values = [score_value(record, field) for record in valid_records]
        normed_by_gate[gate] = normalize_values(values, FUSION_NORMALIZE)

    fused_values = []
    for i, record in enumerate(valid_records):
        normalized = {gate: normed_by_gate[gate][i] for gate in fusion_gates}
        fused = fusion_score_from_normalized(normalized, FUSION_WEIGHTS, fusion_gates)
        record.setdefault('scores', {})['fusion'] = float(fused)
        fused_values.append(float(fused))

    threshold = float(np.quantile(np.asarray(fused_values, dtype=np.float64), 1.0 - float(FUSION_KEEP_TOP)))
    filtered_records = [record for record in valid_records if float(record['scores']['fusion']) >= threshold]
    filtered_records.sort(key=lambda record: float(record['scores']['fusion']), reverse=True)
else:
    raise ValueError(f'FILTER_MODE không hợp lệ: {FILTER_MODE}. Dùng top_quantile, cascade, fusion hoặc all.')
if MAX_SYNTH_RECORDS is not None:
    filtered_records = filtered_records[:MAX_SYNTH_RECORDS]

for record in filtered_records:
    answer = record.get('answer', [])
    if isinstance(answer, str):
        record['answer'] = [answer]
    record['dataset'] = 'vg'  # Route pseudo images qua vg_root=SYNTH_IMAGE_ROOT.
    record['image'] = Path(record['image']).name

print('Score input:', score_input_path)
print('Records input:', scored_json_path)
if cache_files_used:
    print('Cache used:', cache_files_used)
print('Experiment:', EXPERIMENT_NAME)
print('Filter mode:', FILTER_MODE)
print('Score fields:', required_score_fields() if FILTER_MODE != 'all' else '(unused)')
if FILTER_MODE == 'fusion':
    print('Fusion normalize:', FUSION_NORMALIZE)
    print('Fusion weights:', FUSION_WEIGHTS)
    print('Fusion keep_top:', FUSION_KEEP_TOP)
if isinstance(threshold, dict):
    threshold_text = {key: round(value, 6) for key, value in threshold.items()}
else:
    threshold_text = 'None' if threshold is None else f'{threshold:.6f}'
print(f'Input={len(scored_records):,}; threshold={threshold_text}; kept={len(filtered_records):,} ({len(filtered_records)/len(scored_records):.2%})')

In [ ]:
import shutil
required_aokvqa_files = ['train.json', 'val.json', 'answer_list.json']
missing_aokvqa_files = [name for name in required_aokvqa_files if not (AOKVQA_INPUT_ROOT / name).is_file()]
if missing_aokvqa_files:
    raise FileNotFoundError(
        f'A-OKVQA input thiếu {missing_aokvqa_files} trong {AOKVQA_INPUT_ROOT}. '
        'Hãy attach dataset phong2004/a-okvqa.'
    )
if not COCO_INPUT_ROOT.is_dir():
    raise FileNotFoundError(
        f'Không thấy COCO input {COCO_INPUT_ROOT}. '
        'Hãy attach dataset awsaf49/coco-2017-dataset.'
    )

AOKVQA_ROOT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(AOKVQA_INPUT_ROOT, AOKVQA_ROOT, dirs_exist_ok=True)

with (AOKVQA_ROOT / 'train.json').open() as f:
    train_records = json.load(f)
with (AOKVQA_ROOT / 'val.json').open() as f:
    val_records = json.load(f)
with (AOKVQA_ROOT / 'answer_list.json').open() as f:
    answer_list = json.load(f)
print(f'Đã copy A-OKVQA: {AOKVQA_INPUT_ROOT} -> {AOKVQA_ROOT}')
print(f'A-OKVQA train={len(train_records):,}; val={len(val_records):,}')

In [ ]:
def atomic_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)
    os.replace(tmp, path)

# COCO dataset giữ ảnh trong train2017/ và val2017/; chỉ sửa relative path trong bản annotation đã copy.
for record in train_records:
    record['image'] = f'train2017/{Path(record["image"]).name}'
for record in val_records:
    record['image'] = f'val2017/{Path(record["image"]).name}'

atomic_json(AOKVQA_ROOT / 'train.json', train_records)
atomic_json(AOKVQA_ROOT / 'val.json', val_records)
filtered_synth_path = AOKVQA_ROOT / f'{FILTERED_SYNTH_FILE}.json'
atomic_json(filtered_synth_path, filtered_records)
print('Đã ghi annotation; không copy hoặc link ảnh.')

In [ ]:
missing_coco = [
    COCO_INPUT_ROOT / record['image']
    for record in train_records + val_records
    if not (COCO_INPUT_ROOT / record['image']).is_file()
]
missing_synth = [
    SYNTH_IMAGE_ROOT / record['image']
    for record in filtered_records
    if not (SYNTH_IMAGE_ROOT / record['image']).is_file()
]
if missing_coco or missing_synth:
    raise FileNotFoundError(
        f'Missing COCO={len(missing_coco)}, synthetic={len(missing_synth)}; '
        f'ví dụ: {(missing_coco + missing_synth)[:10]}'
    )
print(f'Dùng trực tiếp {len(train_records) + len(val_records):,} A-OKVQA images từ {COCO_INPUT_ROOT}')
print(f'Dùng trực tiếp {len(filtered_records):,} pseudo images từ {SYNTH_IMAGE_ROOT}')

In [ ]:
import yaml

config = {
    'vqa_root': str(COCO_INPUT_ROOT),
    'vg_root': str(SYNTH_IMAGE_ROOT),
    'train_files': ['train', FILTERED_SYNTH_FILE],
    'ann_root': str(AOKVQA_ROOT),
    'dataset_name': 'aokvqa',
    'truncate_train_dataset_to': TRUNCATE_TRAIN_TO,
    'append_rationale_to_answer': False,
    'append_rationale_to_question': False,
    'use_rationale_as_answer': False,
    'use_validation_set_as_test_set': True,
    'pretrained': 'https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_base_capfilt_large.pth',
    'vit': 'base',
    'batch_size_train': BATCH_SIZE_PER_GPU,
    'batch_size_test': BATCH_SIZE_TEST,
    'vit_grad_ckpt': False,
    'vit_ckpt_layer': 0,
    'init_lr': 2e-5,
    'image_size': 480,
    'k_test': 128,
    'inference': 'rank',
    'weight_decay': 0.05,
    'min_lr': 0,
    'max_epoch': MAX_EPOCH,
    'torch_home': '/kaggle/working/torch_home',
    'wandb': False,
    'save_last_only': False,
    'max_checkpoints': 3,
}
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
print(CONFIG_PATH.read_text())

In [ ]:
import torch

assert torch.cuda.is_available(), 'Bật GPU trong Kaggle Settings > Accelerator'
gpu_count = min(MAX_GPUS, torch.cuda.device_count())
for gpu_id in range(gpu_count):
    print(f'cuda:{gpu_id}:', torch.cuda.get_device_name(gpu_id))
print(f'Global batch size={gpu_count * BATCH_SIZE_PER_GPU}')

%cd {REPO_DIR}
!python -c 'import train_vqa; print("train_vqa import OK")'
if _exit_code != 0:
    raise RuntimeError(f'train_vqa import failed với exit code {_exit_code}')

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
resume_args = ''
if RESUME_CHECKPOINT:
    resume_path = Path(RESUME_CHECKPOINT)
    if not resume_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy RESUME_CHECKPOINT: {resume_path}')
    resume_args = f'--resume {resume_path}'
    print('Resume từ checkpoint cụ thể:', resume_path)
elif AUTO_RESUME:
    existing_checkpoints = sorted(OUTPUT_DIR.glob('checkpoint_*.pth'))
    if existing_checkpoints:
        print('Auto-resume từ checkpoint mới nhất trong OUTPUT_DIR:', existing_checkpoints[-1])
    else:
        print('Không có checkpoint trong OUTPUT_DIR, train từ pretrained.')
else:
    resume_args = '--no-resume'
    print('Resume tắt, train từ pretrained.')

%cd {REPO_DIR}
!torchrun --standalone --nproc_per_node={gpu_count} train_vqa.py --config={CONFIG_ARG} --output_dir={OUTPUT_DIR} {resume_args} --seed={SEED}
if _exit_code != 0:
    raise RuntimeError(f'torchrun failed với exit code {_exit_code}')
print('Training và inference A-OKVQA validation hoàn tất.')

In [ ]:
checkpoints = sorted(OUTPUT_DIR.glob('checkpoint_*.pth'))
assert checkpoints, f'Không tìm thấy checkpoint trong {OUTPUT_DIR}'
for checkpoint in checkpoints:
    print(f'{checkpoint.name}: {checkpoint.stat().st_size / 1024**3:.2f} GiB')
print('Student checkpoint:', checkpoints[-1])
print('Prediction:', OUTPUT_DIR / 'result/vqa_result.json')
print('Training log:', OUTPUT_DIR / 'log.txt')
print('Filtered synthetic:', AOKVQA_ROOT / f'{FILTERED_SYNTH_FILE}.json')